# Tutorial: Upload, access and explore your data in Azure Machine Learning

In this tutorial you learn how to:

> * Upload your data to cloud storage
> * Create an Azure Machine Learning data asset
> * Access your data in a notebook for interactive development
> * Create new versions of data assets

The start of a machine learning project typically involves exploratory data analysis (EDA), data-preprocessing (cleaning, feature engineering), and the building of Machine Learning model prototypes to validate hypotheses. This _prototyping_ project phase is highly interactive. It lends itself to development in an IDE or a Jupyter notebook, with a _Python interactive console_. This tutorial describes these ideas.

## Prerequisites

* If you opened this notebook from Azure Machine Learning studio, you need a compute instance to run the code. If you don't have a compute instance, select **Create compute** on the toolbar to first create one.  You can use all the default settings.  

    ![Create compute](./media/create-compute.png)

* If you're seeing this notebook elsewhere, complete [Create resources you need to get started](https://docs.microsoft.com/azure/machine-learning/quickstart-create-resources) to create an Azure Machine Learning workspace and a compute instance.

## Set your kernel

* If your compute instance is stopped, start it now.  
        
    ![Start compute](./media/start-compute.png)

* Once your compute instance is running, make sure the that the kernel, found on the top right, is `Python 3.10 - SDK v2`.  If not, use the dropdown to select this kernel.

    ![Set the kernel](./media/set-kernel.png)

### Download the data used in this tutorial

For data ingestion, the Azure Data Explorer handles raw data in [these formats](https://learn.microsoft.com/azure/data-explorer/ingestion-supported-formats). This tutorial uses this [CSV-format credit card client data sample](https://azuremlexamples.blob.core.windows.net/datasets/credit_card/default_of_credit_card_clients.csv). We see the steps proceed in an Azure Machine Learning resource. In that resource, we'll create a local folder with the suggested name of **data** directly under the folder where this notebook is located.

> [!NOTE]
> This tutorial depends on data placed in an Azure Machine Learning resource folder location. For this tutorial, 'local' means a folder location in that Azure Machine Learning resource. 

1. Select **Open terminal** below the three dots, as shown in this image:

    ![Open terminal](./media/open-terminal.png)

1. The terminal window opens in a new tab. 
1. Make sure you `cd` to the same folder where this notebook is located.  For example, if the notebook is in a folder named **get-started-notebooks**:

    ```
    cd get-started-notebooks    #  modify this to the path where your notebook is located
    ```

1. Enter these commands in the terminal window to copy the data to your compute instance:

    ```
    mkdir data
    cd data                     # the sub-folder where you'll store the data
    wget https://azuremlexamples.blob.core.windows.net/datasets/credit_card/default_of_credit_card_clients.csv
    ```
1. You can now close the terminal window.


[Learn more about this data on the UCI Machine Learning Repository.](https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients)

## Create handle to workspace

Before we dive in the code, you need a way to reference your workspace. You'll create `ml_client` for a handle to the workspace.  You'll then use `ml_client` to manage resources and jobs.

In the next cell, enter your Subscription ID, Resource Group name and Workspace name. To find these values:

1. In the upper right Azure Machine Learning studio toolbar, select your workspace name.
1. Copy the value for workspace, resource group and subscription ID into the code.
1. You'll need to copy one value, close the area and paste, then come back for the next one.

In [7]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# authenticate
credential = DefaultAzureCredential()

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="abb3353d-14ab-4405-8fec-2be226eecc91",
    resource_group_name="BARRIBAL.JOSHUAGEORGE-rg",
    workspace_name="aquagrow",
)

> [!NOTE]
> Creating MLClient will not connect to the workspace. The client initialization is lazy, it will wait for the first time it needs to make a call (this will happen in the next code cell).


## Upload data to cloud storage

Azure Machine Learning uses Uniform Resource Identifiers (URIs), which point to storage locations in the cloud. A URI makes it easy to access data in notebooks and jobs. Data URI formats look similar to the web URLs that you use in your web browser to access web pages. For example:

* Access data from public https server: `https://<account_name>.blob.core.windows.net/<container_name>/<folder>/<file>`
* Access data from Azure Data Lake Gen 2: `abfss://<file_system>@<account_name>.dfs.core.windows.net/<folder>/<file>`

An Azure Machine Learning data asset is similar to web browser bookmarks (favorites). Instead of remembering long storage paths (URIs) that point to your most frequently used data, you can create a data asset, and then access that asset with a friendly name.

Data asset creation also creates a *reference* to the data source location, along with a copy of its metadata. Because the data remains in its existing location, you incur no extra storage cost, and don't risk data source integrity. You can create Data assets from Azure Machine Learning datastores, Azure Storage, public URLs, and local files.

> [!TIP]
> For smaller-size data uploads, Azure Machine Learning data asset creation works well for data uploads from local machine resources to cloud storage. This approach avoids the need for extra tools or utilities. However, a larger-size data upload might require a dedicated tool or utility - for example, **azcopy**. The azcopy command-line tool moves data to and from Azure Storage. Learn more about [azcopy](https://learn.microsoft.com/en-us/azure/storage/common/storage-use-azcopy-v10).

The next notebook cell creates the data asset. The code sample uploads the raw data file to the designated cloud storage resource.  

Each time you create a data asset, you need a unique version for it.  If the version already exists, you'll get an error.  In this code, we're using time to generate a unique version each time the cell is run.

You can also omit the **version** parameter, and a version number is generated for you, starting with 1 and then incrementing from there. In this tutorial, we want to refer to specific version numbers, so we create a version number instead.

In [8]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import time

# update the 'my_path' variable to match the location of where you downloaded the data on your
# local filesystem

my_path = "./data/meanIntervalData.csv"
# set the version number of the data asset to the current UTC time
v1 = time.strftime("%Y.%m.%d.%H%M%S", time.gmtime())


my_data = Data(
    name="mean-dataset",
    version=v1,
    description="Aquagrow mean data set",
    path=my_path,
    type=AssetTypes.URI_FILE,
)

# create data asset
ml_client.data.create_or_update(my_data)

print(f"Data asset created. Name: {my_data.name}, version: {my_data.version}")

Uploading meanIntervalData.csv (< 1 MB): 100%|██████████| 9.77k/9.77k [00:00<00:00, 848kB/s]




Data asset created. Name: mean-dataset, version: 2023.05.04.181155


You can see the uploaded data by selecting **Data** on the left. You'll see the data is uploaded and a data asset is created:

![Image of data section of studio shows uploaded data](./media/access-and-explore-data.png)

This data is named **credit-card**, and in the **Data assets** tab, we can see it in the **Name** column. This data uploaded to your workspace's default datastore named **workspaceblobstore**, seen in the **Data source** column. 

An Azure Machine Learning datastore is a *reference* to an *existing* storage account on Azure. A datastore offers these benefits:

1. A common and easy-to-use API, to interact with different storage types (Blob/Files/Azure Data Lake Storage) and authentication methods.
1. An easier way to discover useful datastores, when working as a team.
1. In your scripts, a way to hide connection information for credential-based data access (service principal/SAS/key).


## Access your data in a notebook

Pandas directly support URIs - this example shows how to read a CSV file from an Azure Machine Learning Datastore:

```
import pandas as pd

df = pd.read_csv("azureml://subscriptions/<subid>/resourcegroups/<rgname>/workspaces/<workspace_name>/datastores/<datastore_name>/paths/<folder>/<filename>.csv")
```

However, as mentioned previously, it can become hard to remember these URIs. Additionally, you must manually substitute all **<_substring_>** values in the **pd.read_csv** command with the real values for your resources. 

You'll want to create data assets for frequently accessed data. Here's an easier way to access the CSV file in Pandas:

> [!IMPORTANT]
> In a notebook cell, execute this code to install the `azureml-fsspec` Python library in your Jupyter kernel:

In [9]:
%pip install -U azureml-fsspec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 32.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 9.7 MB/s eta 0:00:00:00:010:01m
  Attempting uninstall: azureml-dataprep-rslex
    Found existing installation: azureml-dataprep-rslex 2.16.4
    Uninstalling azureml-dataprep-rslex-2.16.4:
      Successfully uninstalled azureml-dataprep-rslex-2.16.4
  Attempting uninstall: azureml-dataprep
    Found existing installation: azureml-dataprep 4.9.5
    Uninstalling azureml-dataprep-4.9.5:
      Successfully uninstalled azureml-dataprep-4.9.5
  Attempting uninstall: azureml-fsspec
    Found existing installation: azureml-fsspec 0.1.0b3
    Uninstalling azureml-fsspec-0.1.0b3:
      Successfully uninstalled azureml-fsspec-0.1.0b3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mltable 1.2.0 requires azureml-dataprep[pa

In [11]:
import pandas as pd

# get a handle of the data asset and print the URI
data_asset = ml_client.data.get(name="mean-dataset", version=v1)
print(f"Data asset URI: {data_asset.path}")

# read into pandas - note that you will see 2 headers in your data frame - that is ok, for now

df = pd.read_csv(data_asset.path)
df.head()

Data asset URI: azureml://subscriptions/abb3353d-14ab-4405-8fec-2be226eecc91/resourcegroups/BARRIBAL.JOSHUAGEORGE-rg/workspaces/aquagrow/datastores/workspaceblobstore/paths/LocalUpload/7251c980c9ce25d1bc38a162a9dbd6f8/meanIntervalData.csv


,ambient_temperature,humidity,dissolved_oxygen,electrical_conductivity,ph_level,temperature
0,34.896598,62.753868,6.307200,9.256473,6.984857,30.950521
1,34.362855,64.693197,6.748143,9.198907,7.089195,30.433486
2,31.620019,73.221110,7.264633,9.026523,7.563556,28.250383
3,30.304183,76.367711,7.208256,8.451398,8.035779,26.484686
4,30.360142,75.203441,7.075773,8.008163,8.565331,26.515291


Read [Access data from Azure cloud storage during interactive development](how-to-access-data-interactive.md) to learn more about data access in a notebook.

## Create a new version of the data asset

You might have noticed that the data needs a little light cleaning, to make it fit to train a machine learning model. It has:

* two headers
* a client ID column; we wouldn't use this feature in Machine Learning
* spaces in the response variable name

Also, compared to the CSV format, the Parquet file format becomes a better way to store this data. Parquet offers compression, and it maintains schema. Therefore, to clean the data and store it in Parquet, use:

In [39]:
# read in data again, this time using the 2nd row as the header
df = pd.read_csv(data_asset.path, header=3)
# rename column
#df.rename(columns={"default payment next month": "default"}, inplace=True)
# remove ID column
df.drop(["sensor", "_time", "result", "table", "_measurement", "_start", "_stop", "Unnamed: 0" ], axis=1, inplace=True)
#df.head()



In [36]:
import numpy as np

# Define the number of data points you want to generate
num_points = 89

# Define the range of the original data
min_weight = 3.5
max_weight = 121.6

# Generate a new set of weight measurements
new_weights = np.linspace(min_weight, max_weight, num_points)

# Add noise to the weight measurements
noise = np.random.normal(0, 1, num_points) * 0.1  # adjust the standard deviation as needed
new_weights += noise

# Apply scaling to the weight measurements
scaling = np.random.uniform(0.9, 1.1, num_points)  # adjust the scaling range as needed
new_weights *= scaling

# Interpolate new weight measurements between existing measurements
old_weights = np.linspace(min_weight, max_weight, 24)
new_weights = np.interp(np.linspace(0, 1, num_points), np.linspace(0, 1, 24), old_weights)
print(new_weights)

new_weights_df = pd.DataFrame({'weight' : new_weights})

print(new_weights_df)



[  3.5          4.84204545   6.18409091   7.52613636   8.86818182
  10.21022727  11.55227273  12.89431818  14.23636364  15.57840909
  16.92045455  18.2625      19.60454545  20.94659091  22.28863636
  23.63068182  24.97272727  26.31477273  27.65681818  28.99886364
  30.34090909  31.68295455  33.025       34.36704545  35.70909091
  37.05113636  38.39318182  39.73522727  41.07727273  42.41931818
  43.76136364  45.10340909  46.44545455  47.7875      49.12954545
  50.47159091  51.81363636  53.15568182  54.49772727  55.83977273
  57.18181818  58.52386364  59.86590909  61.20795455  62.55
  63.89204545  65.23409091  66.57613636  67.91818182  69.26022727
  70.60227273  71.94431818  73.28636364  74.62840909  75.97045455
  77.3125      78.65454545  79.99659091  81.33863636  82.68068182
  84.02272727  85.36477273  86.70681818  88.04886364  89.39090909
  90.73295455  92.075       93.41704545  94.75909091  96.10113636
  97.44318182  98.78522727 100.12727273 101.46931818 102.81136364
 104.15340909 10

In [47]:
df['weight'] = new_weights_df
print(df.head())
# write file to filesystem
df.to_parquet("./data/clean-mean-dataset.parquet")

   ambient_temperature  dissolved_oxygen      do_volt      ec_volt   
0            34.694102          6.602502  1452.121951  1687.787140  \
1            34.362855          6.748143  1466.727829  1660.145260   
2            31.620019          7.264633  1473.612557  1570.292496   
3            30.304183          7.208256  1373.742726  1424.099541   
4            30.360142          7.075773  1350.128440  1350.139144   

   electrical_conductivity   humidity  ph_level    ph_voltge  temperature   
0                 9.299618  63.540697  7.204389  1463.725055    30.761641  \
1                 9.198907  64.693197  7.089195  1484.169725    30.433486   
2                 9.026523  73.221110  7.563556  1399.980092    28.250383   
3                 8.451398  76.367711  8.035779  1316.169985    26.484686   
4                 8.008163  75.203441  8.565331  1222.185015    26.515291   

     weight  
0  3.500000  
1  4.842045  
2  6.184091  
3  7.526136  
4  8.868182  


This table shows the structure of the data in the original **default_of_credit_card_clients.csv** file .CSV file downloaded in an earlier step. The uploaded data contains 23 explanatory variables and 1 response variable, as shown here:

|Column Name(s) | Variable Type  |Description  |
|---------|---------|---------|
|X1     |   Explanatory      |    Amount of the given credit (NT dollar): it includes both the individual consumer credit and their family (supplementary) credit.    |
|X2     |   Explanatory      |   Gender (1 = male; 2 = female).      |
|X3     |   Explanatory      |   Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).      |
|X4     |   Explanatory      |    Marital status (1 = married; 2 = single; 3 = others).     |
|X5     |   Explanatory      |    Age (years).     |
|X6-X11     | Explanatory        |  History of past payment. We tracked the past monthly payment records (from April to September  2005). -1 = pay duly; 1 = payment delay for one month; 2 = payment delay for two months; . . .; 8 = payment delay for eight months; 9 = payment delay for nine months and above.      |
|X12-17     | Explanatory        |  Amount of bill statement (NT dollar) from April to September  2005.      |
|X18-23     | Explanatory        |  Amount of previous payment (NT dollar) from April to September  2005.      |
|Y     | Response        |    Default payment (Yes = 1, No = 0)     |

Next, create a new _version_ of the data asset (the data automatically uploads to cloud storage):

> [!NOTE]
>
> This Python code cell sets **name** and **version** values for the data asset it creates. As a result, the code in this cell will fail if executed more than once, without a change to these values. Fixed **name** and **version** values offer a way to pass values that work for specific situations, without concern for auto-generated or randomly-generated values.


In [1]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import time

# Next, create a new *version* of the data asset (the data is automatically uploaded to cloud storage):
v3 = "_cleaned-with-weight"
my_path = "./data/clean-mean-dataset-with-weight.parquet"

# Define the data asset, and use tags to make it clear the asset can be used in training

my_data = Data(
    name="mean-dataset-with-weight",
    version=v3,
    description="Aquagrow Mean Dataset",
    tags={"training_data": "true", "format": "parquet"},
    path=my_path,
    type=AssetTypes.URI_FILE,
)

## create the data asset

my_data = ml_client.data.create_or_update(my_data)

print(f"Data asset created. Name: {my_data.name}, version: {my_data.version}")

NameError: name 'ml_client' is not defined

The cleaned parquet file is the latest version data source. This code shows the CSV version result set first, then the Parquet version:

In [23]:
import pandas as pd

# get a handle of the data asset and print the URI
data_asset_v1 = ml_client.data.get(name="mean-dataset", version=v1)
data_asset_v2 = ml_client.data.get(name="mean-dataset", version=v2)

# print the v1 data
print(f"V1 Data asset URI: {data_asset_v1.path}")
v1df = pd.read_csv(data_asset_v1.path)
print(v1df.head(5))

# print the v2 data
print(
    "_____________________________________________________________________________________________________________\n"
)
print(f"V2 Data asset URI: {data_asset_v2.path}")
v2df = pd.read_parquet(data_asset_v2.path)
print(v2df.head(5))

V1 Data asset URI: azureml://subscriptions/abb3353d-14ab-4405-8fec-2be226eecc91/resourcegroups/barribal.joshuageorge-rg/workspaces/aquagrow/datastores/workspaceblobstore/paths/LocalUpload/5419deedf5f20600177e3c61dce4ece5/meandatatest.csv
  #datatype   string   long                dateTime:RFC3339   
0    #group    false  false                            true  \
1  #default  _result    NaN                             NaN   
2       NaN   result  table                          _start   
3       NaN      NaN      0  2023-04-04T03:23:00.945164041Z   
4       NaN      NaN      0  2023-04-04T03:23:00.945164041Z   

               dateTime:RFC3339.1              dateTime:RFC3339.2   
0                            true                           false  \
1                             NaN                             NaN   
2                           _stop                           _time   
3  2023-05-04T03:23:00.945164041Z  2023-05-04T03:23:00.945164041Z   
4  2023-05-04T03:23:00.945164041Z     

## Next steps

Read [Create data assets](https://learn.microsoft.com/azure/machine-learning/how-to-create-data-assets) for more information about data assets.

Read [Create datastores](https://learn.microsoft.com/azure/machine-learning/how-to-datastore) to learn more about datastores.

Continue with tutorials to learn how to develop a training script.

> [Model development on a cloud workstation](https://learn.microsoft.com/azure/machine-learning/tutorial-cloud-workstation)